# W03 - Latent Dirichlet Allocation with Detailed Research

In [1]:
# Import libraries
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import random
import gensim
import time
import string
from nltk.corpus import stopwords
from gensim.utils import simple_preprocess
from gensim.parsing.preprocessing import STOPWORDS
salaries = pd.read_csv('Salaries_v4_202412161100.csv')
# salaries = pd.read_csv('Salaries_v3_202410141100.csv')
# salaries = pd.read_csv('Salaries_v3_202408271150.csv')
# salaries = pd.read_csv('Salaries_v3_202405031235.csv')
salaries = salaries.drop_duplicates('id').reset_index()
salaries = salaries.drop('index',axis=1)

In [2]:
salaries.head()

,id,found_country,title,position,employment_type,company,company_score,edu_degrees,edu_degrees_major,working_year,education_score,ms_counts,skill_counts,main_skills,skills,amount_usd
0,0000083c-7054-4a2b-b675-6ac664c66bfb,United States,"Software Developer II at Audible, Inc.",Software Developer II,Full-time,"Audible, Inc.",8.9,"HIGH_SCHOOL,MASTERS,UNDERGRADUATE","Bachelor’s Degree, Computer Science,High Schoo...",11,8.1,21,20,"Ajax, C, C++, CSS, Data Engineering, HTML, HTM...","ajax, c, c++, cascading style sheets (css), cs...",192000
1,00013847-ecf1-4a5a-ba44-16475dc28eba,United States,Retail Associate at Converse,Retail Associate,Full-time,NaN,NaN,UNDERGRADUATE,NaN,5,NaN,14,10,"Communication, Customer Service, Deadline Mana...","communication, customer service, employee trai...",38000
2,00018332-5b5d-4c23-88f8-1c2cdc133e28,United Kingdom,Test Engineer at Sky,Test Engineer,Full-time,NaN,NaN,UNDERGRADUATE,"Bachelor of Science (BSc), Computer Software E...",12,7.0,15,20,"Agile, Attention to Detail, Automated Testing,...","agile methodologies, api testing, communicatio...",66191
3,000c1054-ab28-4c4d-90b0-fa4b1ed31a2a,United States,Hardware Engineer at Google,Hardware Engineer,NaN,Google,8.7,MASTERS,"Master of Science (MS), Computer Engineering",13,8.9,9,20,"C, Circuit Design, Debugging, Embedded Softwar...","asic, c, computer architecture, debugging, emb...",341000
4,00145b03-e286-4bdc-9063-ed5d2095306a,United States,BI Engineer @ Amazon | MS,BIE II,NaN,Amazon,8.4,MASTERS,"Master of Science - MS, Data Analytics",3,8.5,6,9,"JavaScript, Marketing Research, Microsoft Exce...","javascript, microsoft excel, microsoft office,...",185000


## Examining Stopwords

### English languages stopwords of 'nltk.corpus.stopwords'

In [3]:
print(set(stopwords.words('english')))

{'hasn', 'through', 'being', 'itself', 'same', 'hers', "hadn't", 'too', 't', 'y', "won't", 'were', 'so', "mightn't", "it's", 'was', 'for', "you're", 'them', "you've", 'or', "don't", 'mustn', 'this', 'herself', 'hadn', "weren't", 'didn', 'with', 'have', 'don', 'at', 'himself', 'been', "mustn't", 'theirs', 'other', "wouldn't", 'no', 'me', 'while', 'haven', 'not', 'll', 'your', 'more', 'any', 'only', "doesn't", 'into', 'an', "shouldn't", 'until', "that'll", 'which', 'both', 'mightn', "couldn't", 'doing', 'ours', 'who', 'that', 'in', 'of', 'between', 'm', "should've", 'its', 'o', 'how', 'where', 'isn', 'you', 'from', "hasn't", 'a', "didn't", 'against', 'to', 'down', 'here', 'are', 'further', 'nor', 'few', "needn't", 'won', 'up', 'ourselves', 'he', 's', 'couldn', 'above', 'off', 'own', 'd', 'the', 'aren', 'what', 'these', 'needn', 'wasn', 'will', 'yourself', 'did', 'themselves', 'by', 'those', "isn't", 'wouldn', 'has', 'each', 'yourselves', 'weren', 'shouldn', 'myself', 'i', 'whom', 'can', 

### English language stopwords of 'gensim.parsing.preprocessing.STOPWORDS'

In [4]:
print(STOPWORDS)

frozenset({'through', 'without', 'meanwhile', 'empty', 'wherein', 'hers', 'too', 'besides', 'were', 'beside', 'thereafter', 'them', 'various', 'hereafter', 'give', 'via', 'less', 'himself', 'no', 'nobody', 'latter', 'until', 'five', 'get', 'hereupon', 'ours', 'who', 'that', 'of', 'part', 'used', 'throughout', 'here', 'few', 'describe', 'up', 'above', 'will', 'whither', 'yourself', 'quite', 'become', 'next', 'by', 'twelve', 'please', 'thru', 'top', 'due', 'noone', 'whom', 'very', 'using', 'under', 'everyone', 'when', 'one', 'eight', 'him', 'such', 'together', 'if', 'ever', 'she', 'everything', 'because', 'then', 'perhaps', 'km', 'during', 'amongst', 'alone', 'am', 'anywhere', 'their', 'four', 'his', 'since', 'below', 'six', 'being', 'itself', 'seemed', 'along', 'same', 'well', 'thereby', 'really', 'system', 'for', 'or', 'with', 'don', 'at', 'been', 'made', 'whereby', 'keep', 'hereby', 'though', 'either', 'must', 'three', 'only', 'an', 'which', 'couldnt', 'both', 'mine', 'back', 'co', 'n

### All text characters under various categories from built-in 'string' library

In [5]:
print("-------- STRING --------")
print("ASCII LETTERS ->", string.ascii_letters)
print("ASCII LOWERCASE ->", string.ascii_lowercase)
print("ASCII UPPERCASE ->", string.ascii_uppercase)
print("DIGITS ->", string.digits)
print("HEXDIGITS ->", string.hexdigits)
print("OCTDIGITS ->", string.octdigits)
print("PRINTABLE ->", string.printable)
print("PUNCTUATION ->", string.punctuation)
print("WHITESPACE ->", string.whitespace)

-------- STRING --------
ASCII LETTERS -> abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ
ASCII LOWERCASE -> abcdefghijklmnopqrstuvwxyz
ASCII UPPERCASE -> ABCDEFGHIJKLMNOPQRSTUVWXYZ
DIGITS -> 0123456789
HEXDIGITS -> 0123456789abcdefABCDEF
OCTDIGITS -> 01234567
PRINTABLE -> 0123456789abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~ 	

PUNCTUATION -> !"#$%&'()*+,-./:;<=>?@[\]^_`{|}~
WHITESPACE ->  	



## Preprocessing

### Process the null titles and companies to make them represented as empty strings

In [6]:
salaries['title'] = salaries['title'].apply(lambda x: '' if type(x) == float else x)
salaries['company'] = salaries['company'].apply(lambda x: '' if type(x) == float else x)

### Manipulate the specific texts and obtain cleaner ones

In [7]:
def manipulate_text(text):
    return text.replace('sr. ','senior ').replace('sr ','senior ').replace(' sr ',' senior ') \
        .replace(' sr. ',' senior ').replace('sr.data','senior data') \
        .replace(',','').replace('i̇','i') \
        .replace(' ex ',' ').replace('ex-','').replace('ex - ','') \
        .replace('𝐀𝐦𝐚𝐳𝐨𝐧','amazon').replace('𝐔𝐒𝐂','usc') \
        .replace("we're hiring",'').replace('bellevue wa','bellevue') \
        .replace('aroraakshit.github.io','githubio') \
        .replace('https://www.buildzoom.com/contractor/brend-renovation-corp','buildzoom').replace('j p morgan','jpmorgan') \
        .replace('j.p. morgan','jpmorgan').replace('jp morgan chase','jpmorgan').replace('jp morgan','jpmorgan') \
        .replace('fisker inc','fisker incorporation').replace('us & overseas','usoverseas').replace('2u','toyou') \
        .replace('mental mastery p. a.','mental mastery').replace('p electrical engineer','electrical engineer') \
        .replace('scientist@salesforce','scientist salesforce').replace('cs@northeastern','cs northeastern') \
        .replace('engineer@linkedin','engineer linkedin').replace('engineer@amazon','engineer amazon') \
        .replace('intern@alibaba','intern alibaba').replace('ms.statistics@uiuc','ms statistic uiuc') \
        .replace('engineer@oracle','engineer oracle').replace('scientist@walmart','scientist walmart') \
        .replace('power bi','powerbi').replace('intellij idea','intellijidea').replace('uh alum','alum') \
        .replace('big data','bigdata').replace('beautiful soup','beautifulsoup').replace('capital one','capitalone') \
        .replace('va boston','vaboston').replace('frito lay','fritolay').replace('final cut pro','finalcutpro') \
        .replace('final cut','finalcut').replace('indesign cc','indesigncc') \
        .replace('programme','program').replace('testng','testing').replace('modelling','modeling').replace('plannig','planning') \
        .replace('hardworkig','hardworking').replace('évaluation','evaluation').replace('telemarkting','telemarketing') \
        .replace('anaylisis','analysis') \
        .replace('qa ',' quality assurance ').replace('nbc10','nbcten').replace('dr horton','drhorton') \
        .replace('engineer 2','engineer').replace('developer 3','developer') \
        .replace('cluster(galera','cluster galera').replace('engineer(vulnerability','engineer vulnerability') \
        .replace('system/2(document','systemtwo document') \
        .replace('construction/test/troubleshoot','construction test troubleshoot').replace('player/builder','player builder') \
        .replace('a/b testing','abtesting').replace('ab test','abtesting').replace('illustrator/graphic','illustrator graphic') \
        .replace('manager/program','manager program').replace('c/c++','c cplusplus').replace('java/python','java python') \
        .replace('development/engineer','development engineer').replace('tutoring/teaching','tutoring teaching') \
        .replace('analyst/scientist','analyst scientist').replace('appdynamics/cisco','appdynamics cisco') \
        .replace('proliant/windows/linux','proliant windows linux').replace('data/software','data software') \
        .replace('ml/database','ml database').replace('internals/big','internals big').replace('analyst/network','analyst network') \
        .replace('counselor/certified','counselor certified').replace('practitioner/certified','practitioner certified') \
        .replace('digital/pre-press/packaging','digital prepress packaging').replace('windows/linux','windows linux') \
        .replace('toyou/trilogy','toyou trilogy').replace('uv/vis','uv vis').replace('development/ml','development ml') \
        .replace('docker/ansible/github','docker ansible github').replace('actions/linux','actions linux') \
        .replace('assurance/automation','assurance automation').replace('tester/data','tester data') \
        .replace('learning/computer','learning computer').replace('mohawk/hispanic','mohawk hispanic') \
        .replace('hydrologist/geotechnical/civil','hydrologist geotechnical civil').replace('youtube/google','youtube google') \
        .replace('nursing/careplanning','nursing careplanning').replace('recruitment/retention','recruitment retention') \
        .replace('producer/product','producer product').replace('scientist/subject','scientist subject') \
        .replace('scientist/machine','scientist machine').replace('amazon/imdb','amazon imdb').replace('at&t/ccie','att ccie') \
        .replace('autocad/mathcad/risa/etabs/bluebeam','autocad mathcad risa etabs bluebeam') \
        .replace('tech/boilermaker','tech boilermaker').replace('dietitian/certified','dietitian certified') \
        .replace('medical/clinical','medical clinical').replace('assistant/administrative','assistant administrative') \
        .replace('hardware/software','hardware software').replace('price/promo/markdown','price promo markdown') \
        .replace('civil/structural/transportation','civil structural transportation') \
        .replace('iaas/paas/saas','iaas paas saas').replace('manager/ansible','manager ansible') \
        .replace('git/ci-cd/agile','git cicd agile').replace('campus/fabric/sd-wan','campus fabric sdwan') \
        .replace('administrator/ts-sci','administrator tssci').replace('implementing/managing','implementing managing') \
        .replace('aaa/siem/ids/pki/tacacs+/radius/kerberos','aaa siem ids pki tacacsplus radius kerberos') \
        .replace('ios/asa/juniper','ios asa juniper').replace('snmp/syslog/qos/ntp/tftp','snmp syslog qos ntp tftp') \
        .replace('mode/multi','mode multi').replace('mode)/termination/crimping','mode termination crimping') \
        .replace('rip/ospf/is-is/eigrp/bgp','rip ospf isis eigrp bgp').replace('mpls/gre/ipsec','mpls gre ipsec') \
        .replace('vlan/arp/stp/etherchannel','vlan arp stp etherchannel').replace('hse/hro/scissor','hse hro scissor') \
        .replace('ipv4/ipv6/nat/dns/dhcp','ipvfour ipvsix nat dns dhcp').replace('lift/cherry','lift cherry') \
        .replace('python/javascript/bash/powershell','python javascript bash powershell') \
        .replace('machine/deep','machine deep').replace('excel/vba','excel vba').replace('aix/linux','aix linux') \
        .replace('database/system','database system').replace('physical/logical','physical logical') \
        .replace('products/services','products services').replace('tracing/tkprof','tracing tkprof') \
        .replace('aesthetician/esthetician','aesthetician esthetician').replace('setup/troubleshooting','setup troubleshooting') \
        .replace('oracle/postgresql/mysql','oracle postgresql mysql').replace('awk/sed/grep','awk sed grep') \
        .replace('devops/aws/oracle/mysql/azure','devops aws oracle mysql azure').replace('manager/operations','manager operations') \
        .replace('bartending/serving','bartending serving') \
        .replace('rsqltableauexcelerp/sapetlkpi','r sql tableau excel erp sap etl kpi') \
        .replace('intelligence/machine','intelligence machine').replace('subcontractor/crew','subcontractor crew') \
        .replace('receiver/transmitter','receiver transmitter').replace('python/django','python django') \
        .replace('backup/restore','backup restore').replace('import/export','import export') \
        .replace('spring/hibernate','spring hibernate').replace('compliance/operations','compliance operations') \
        .replace('analyst/engineer','analyst engineer').replace('designer/creative','designer creative') \
        .replace('sd/sdio','sd sdio').replace('spi/i²c/uart','spi isquarec uart').replace('investor/real','investor real') \
        .replace('technologies/certified','technologies certified').replace('hana/sap','hana sap') \
        .replace('bo/qlik/aws/snowflake','bo qlik aws snowflake').replace('embedded/firmware','embedded firmware') \
        .replace('nutritionist/dietitian','nutritionist dietitian').replace('analytical/research','analytical research') \
        .replace('sensitivity/awareness','sensitivity awareness').replace('transfer/few-shot/meta','transfer fewshot meta') \
        .replace('mining/cleansing/modeling/visualization','mining cleansing modeling visualization') \
        .replace('planner/traffic','planner traffic') \
        .replace('athlete||sdr||b2b','athlete sdr btob').replace('sales||saas||tech','sales saas tech') \
        .replace('server|sap|ssis/ssas|etl|tableau|powerbi|teradata|data','server sap siss ssas etl tableau powerbi teradata data') \
        .replace('azure|aws|kubernetes|microservice|ai','azure aws kubernetes microservice ai') \
        .replace('learning|python|iac|automation|containers|cicd','learning python iac automation containers cicd') \
        .replace('nurse|dialysis|home','nurse dialysis home').replace('executive|saas','executive saas') \
        .replace('owner|bookkeeper|payroll','owner bookkeeper payroll').replace('processing|financial','processing financial') \
        .replace('reporting|quickbooks','reporting quickbooks').replace('python|ms','python ms') \
        .replace('engineer|dae@neu','engineer dae neu').replace('persianenglish&dari','persian english dari') \
        .replace('amazon｜product','amazon product') \
        .replace('architectconsultant','architect consultant').replace('hivepigyarn','hive pig yarn') \
        .replace('cisspgctigmongpengcihceh','cissp gcti gmon gpen gcih ceh').replace('pycharmjboss','pycharm jboss') \
        .replace('workstationjmeterfortify','workstation jmeter fortify').replace('scansonar','scan sonar') \
        .replace('electronicdc','electronic dc').replace('electronicconstruction','electronic construction') \
        .replace('gitjenkinsudeploysplunkpostmansoapui','git jenkins udeploy splunk postman soapui') \
        .replace('serversunixlinuxwinscpputtymobaxterm','servers unix linux winscp putty mobaxterm') \
        .replace('studioeclipsespring','studio eclipse spring').replace('rallyjirawebexskype','rally jira webex skype') \
        .replace('developerrobo3ttoadkibana','developer robothreet toad kibana') \
        .replace('pythonsqlsparkgoogle','python sql spark google').replace('cloudlooker','cloud looker') \
        .replace('googlecloudcertified','google cloud certified').replace('systempmp','system pmp') \
        .replace('java/j2eepythonmicroservicespivotal','java jtwoee python microservices pivotal') \
        .replace('foundryspringsql','foundry spring sql').replace('ericssoncalifornia','ericsson california') \
        .replace('inc.','incorporation').replace('co.','company') \
        .replace('salesforce.org','salesforce').replace('sales force','salesforce').replace('first aid','firstaid') \
        .replace('black box','blackbox').replace('white box','whitebox') \
        .replace('front-end','frontend').replace('front end','frontend').replace('p and l','pandl') \
        .replace('back-end','backend').replace('back end','backend').replace('ruby on rails','rubyonrails') \
        .replace('voice over ip','voiceoverip').replace('internet of things','internetofthings') \
        .replace('javaserver pages','javaserverpages').replace('java server pages','javaserverpages') \
        .replace('go-to-market','market') \
        .replace('parent-teacher','parent teacher').replace('spanish-speaking','spanish speaking') \
        .replace('gardaworld-site','gardaworld site').replace('singer-songwriter','singer songwriter') \
        .replace('walmart-labs','walmart labs').replace('president-icm','president icm') \
        .replace('certification-hardware','certification hardware').replace('sagemaker-groundtruth','sagemaker groundtruth') \
        .replace('model-view-controller','modelviewcontroller').replace('cost-effective','costeffective') \
        .replace('object-oriented','objectoriented').replace('service-oriented','serviceoriented') \
        .replace('field-programmable','field programmable').replace('user-centered','usercentered') \
        .replace('cross-functional','crossfunctional').replace('behavior-driven','behaviordriven') \
        .replace('very-large-scale','verylargescale').replace('random-access','randomaccess') \
        .replace('cyber-physical','cyberphysical').replace('application-specific','applicationspecific') \
        .replace('time-efficient','timeefficient').replace('cloud-native','cloudnative') \
        .replace('executive-level','executivelevel').replace('domain-driven','domaindriven') \
        .replace('decision-making','decisionmaking').replace('medical-surgical','medicalsurgical') \
        .replace('test-driven','testdriven') \
        .replace('chick-fil-a-franchise','chickfila franchise') \
        .replace('grc/irm-secops','grc irm secops') \
        .replace('meta(whatsapp)','meta whatsapp').replace('pi(python)','pi python').replace('oracle(oci)','oracle oci') \
        .replace('iit(bhu)','iit bhu').replace('interpretation(english/chinese)','interpretation english chinese') \
        .replace('ruby on rails','ruby rails').replace('system on chip','system chip') \
        .replace('info sec','info').replace('test‚äìdriven','test ai driven') \
        .replace('u.s.','united states').replace('.com','') \
        .replace('.js','js').replace('t-sql','tsql').replace('pl/sql','plsql') \
        .replace('tcp/ip','tcpip').replace('ui/ux','uiux').replace('ci/cd','cicd').replace('ar/vr','arvr') \
        .replace('bi/data','bi data').replace('ai/ml','ai ml').replace('analyst/data','analyst data') \
        .replace('html/css','html css').replace('wifi 6/6e','wifi').replace('cad/cam','cad cam') \
        .replace('python/c/c++','python c cplusplus').replace('java/j2ee','java jtwoee') \
        .replace('.net','dotnet').replace('c++','cplusplus').replace('c#','csharp').replace('brf+','brfplus') \
        .replace('sql*net/net','sqlnet').replace('sql*plus','sqlplus').replace('sql*loader','sqlloader') \
        .replace('l2/l3','l2 l3').replace('p&l','profit loss').replace('apis','api') \
        .replace('gd&t','geometric dimensioning tolerancing').replace('d&o','directors officers') \
        .replace('r&d','research development').replace('v&v','verification validation') \
        .replace('m&a','mergers acquisitions') \
        .replace('exam fm','examfm').replace('exam p','examp').replace(' p spice',' pspice') \
        .replace('html 5','html').replace('html5','html').replace('web 2.0','web') \
        .replace('unity3d','unity 3d').replace('cocos2d','cocostwod') \
        .replace('angular 2','angular').replace('python 3','python') \
        .replace('windows 7','windows').replace('windows 8.1','windows').replace('windows 10','windows') \
        .replace('windows xp','windows').replace('microsoft 365','microsoft').replace('office 365','office') \
        .replace('mac os x','macos').replace('mac os','macos').replace('os x','macos') \
        .replace('vs code','visual studio code').replace('m code','mcode').replace('matlap','matlab') \
        .replace('1st','first').replace('2nd','second').replace('3rd','third').replace('4th','fourth') \
        .replace('2d','twodimension').replace('3d','threedimension').replace('4d','fourdimension') \
        .replace('3g','threeg').replace('4g','fourg').replace('5g','fiveg') \
        .replace('b2b','btob').replace('h.264','htwosixfour').replace('ec2','ectwo').replace('s3','sthree') \
        .replace('db2','dbtwo').replace('hl7','hlseven').replace('ipv4','ipvfour').replace('ipv6','ipvsix') \
        .replace('j2ee','jtwoee').replace('5s','fives').replace('3m','threem').replace('m3','mthree') \
        .replace('ss7','ssseven').replace('robo3t','robothreet').replace('mpeg-4','mpegfour').replace('mpeg4','mpegfour') \
        .replace('mpeg-2','mpegtwo').replace('mpeg2','mpegtwo').replace('m5','mfive').replace('l10n','ltenn') \
        .replace('rg6','rgsix').replace('a+','aplus').replace('a +','aplus').replace('r/3','rthree') \
        .replace('p4','pfour').replace('p6','psix').replace('p/1','pone').replace('fm/2','fmtwo') \
        .replace('x86_64','xeightysixsixtyfour').replace('x86','xeightysix').replace('mfe/3f','mfethreef') \
        .replace('s/4hana','sfourhana') \
        .replace('2x','').replace('2 x','').replace('hold 3','').replace('2400v','').replace('15kv','').replace('10g/11g','') \
        .replace('401k','').replace('sap2000','sap').replace('2005/2000','').replace('2000-2008','').replace('10-hour','') \
        .replace('m1','').replace('k-12','').replace('13c','').replace('11g','') \
        .replace('4.0','').replace('5.0','').replace("'23",'').replace("'21",'').replace('7+','').replace('#1','') \
        .replace('1','').replace('2','').replace('3','').replace('4','').replace('5','') \
        .replace('6','').replace('7','').replace('8','').replace('9','').replace('0','') \
        .replace(' my ',' ').replace(' him ',' ').replace(' her ',' ').replace(' with ',' ').replace(' from ',' ') \
        .replace(' are ',' ').replace(' who ',' ').replace(' for ',' ').replace(' the ',' ').replace(' and ',' ') \
        .replace(' in ', ' ').replace(' of ',' ').replace(' to ',' ').replace(' under ',' ') \
        .replace(' i ', ' ').replace(' ii ',' ').replace(' iii ',' ').replace(' l ',' ').replace(' ll ',' ') \
        .replace(' at ',' ').replace(' as ',' ').replace(' on ',' ').replace(' a ',' ').replace(' an ',' ') \
        .replace(' ww ',' ') \
        .replace('.','').replace(';','').replace('#','').replace('+','').replace('/','').replace('\\','') \
        .replace("'",'').replace('"','').replace('-','').replace('–','').replace('—','').replace('=','') \
        .replace('|','').replace('&','').replace('‘','').replace(':','').replace('®','').replace('!','') \
        .replace('?','').replace('_','').replace('’','').replace('“','').replace('”','').replace('@','') \
        .replace('(','').replace(')','').replace('[','').replace(']','').replace('{','').replace('}','') \
        .replace('•','').replace('┃','').replace('♦','').replace('★','').replace('｜','').replace('▪️','') \
        .replace('♣','').replace('➤','').replace('⚙','').replace('™','') \
        .replace('☁','').replace('	','') \
        .replace('\uf8ff','').replace('🖼','').replace('🏘','') \
        .replace('☁️','').replace('📈','').replace('📊','').replace('🎨','').replace('💵','').replace('✨','') \
        .replace('🐝','').replace('🎯','').replace('🪖','').replace('🔜','').replace('💻','').replace('🦘','').replace('👨🏾‍','') \
        .replace('😊','').replace('👇','').replace('🏠','').replace('🟠','').replace('⚪️','').replace('🐾','').replace('🌐','') \
        .replace('✏️','').replace('💖','').replace('🫶🏾','').replace('👨‍🎓','').replace('📍','').replace('🏆','').replace('🥰','') \
        .replace('💡','').replace('❤️','').replace('💼','').replace('🎙️','').replace('🤝','').replace('👩🏽‍🎓','').replace('🎉','') \
        .replace('🔎','').replace('🤖','').replace('⚙️','').replace('🎟️','').replace('🏳️‍🌈','').replace('🪴','').replace('📉','') \
        .replace('⚡️','').replace('✍️','').replace('🤸🏽','').replace('📢','').replace('👨🏻‍','').replace('🍎','').replace('🎮','') \
        .replace('🧑🏻‍','').replace('🪬','').replace('🚀','').replace('™️','').replace('🚗','').replace('🧠','').replace('🏀','') \
        .replace('💰','').replace('🌎','').replace('⭐','').replace('🎓','').replace('💁','').replace('💊','').replace('🔍','') \
        .replace('📲','').replace('👩‍','').replace('🧡','').replace('🚨','').replace('🔋','') \
        .replace(' 🏿','').replace('   ',' ').replace('  ',' ').strip()

### Translate specific words and/or expressions from other languages into English

In [8]:
### Languages included: Turkish, Spanish, Portuguese, German, French, Dutch, Russian, Polish, Chinese, Korean, Indian
def translate_to_english(text):
    return text.replace('optimisation','optimization') \
        .replace('yetenek olgunluk model entegrasyonu','capability maturity model integration') \
        .replace('nesne yönelimli programlama','object oriented programming') \
        .replace('topluluk önünde konuşma','public speaking') \
        .replace('yapay sinir ağları','artificial neural networks') \
        .replace('işletim sistemleri','operating systems') \
        .replace('nesnelerin interneti','internetofthings') \
        .replace('proje yönetimi','project management') \
        .replace('programlama dilleri','programming languages') \
        .replace('matematik modelleme','mathematical modeling') \
        .replace('tasarımcı düşünce','designer thinking') \
        .replace('analitik beceriler','analytical skills') \
        .replace('endüstriyel tasarım','industrial design') \
        .replace('kavramsal tasarım','conceptual design') \
        .replace('makine öğrenimi','machine learning') \
        .replace('müşteri hizmetleri','customer service') \
        .replace('öğrenme yetenekleri','learning abilities') \
        .replace('piyasa araştırması','market research') \
        .replace('tasarım örüntüleri','design patterns') \
        .replace('assembly dili','assembly language') \
        .replace('ürün geliştirme','product development') \
        .replace('ürün tasarımı','product design') \
        .replace('yedekleme planlaması','backup planning') \
        .replace('veritabanı geliştirme','database development') \
        .replace('veri yapıları','data structures') \
        .replace('veri bilimi','data science') \
        .replace('veri analizi','data analysis') \
        .replace('agile metotları','agile methods') \
        .replace('yabancı diller','foreign languages') \
        .replace('yapay zeka','artificial intelligence') \
        .replace('yazılım tasarımı','software design') \
        .replace('ekip çalışması','teamwork') \
        .replace('gıda güvenliği','food safety') \
        .replace('yiyecek endüstrisi','food industry') \
        .replace('yiyecek hazırlama','food preparation') \
        .replace('doğrudan satış','direct sales') \
        .replace('algoritmalar','algorithms') \
        .replace('geliştirme','development') \
        .replace('programlama','programming') \
        .replace('araştırma','research') \
        .replace('yazılım','software') \
        .replace('ingilizce','english') \
        .replace('eğitim','education') \
        .replace('kahve','coffee') \
        .replace('kasiyer','cashier') \
        .replace('satış','sales') \
        .replace('conjunto de datos e información sobre la eficacia de la atención médica',
                 'data information set health care effectiveness') \
        .replace('setor de produção de petróleo e gás natural','oil natural gas production sector') \
        .replace('administración y dirección de empresas','business administration management') \
        .replace('aptitudes de laboratorio','laboratory skills') \
        .replace('conocimientos informáticos','computer skills') \
        .replace('arquitectura informática','computer architecture') \
        .replace('depuración de programas','program debugging') \
        .replace('desarrollo de productos','product development') \
        .replace('estratégia empresarial','business strategy') \
        .replace('gestión de proyectos','project management') \
        .replace('razonamiento deductivo','deductive reasoning') \
        .replace('instrumentación electrónica','electronic instrumentation') \
        .replace('lanzamiento de productos','product launch') \
        .replace('planificación estratégica','strategic planning') \
        .replace('procesamiento de señales','signal processing') \
        .replace('pruebas de validación','validation test') \
        .replace('validación y verificación','validation verification') \
        .replace('reanimación cardiopulmonar','cardiopulmonary resuscitation') \
        .replace('venta farmacéutica','pharmaceutical sales') \
        .replace('programación de citas','appointment scheduling') \
        .replace('soporte vital básico','basic life support') \
        .replace('educación del paciente','patient education') \
        .replace('automação de testes','test automation') \
        .replace('cuidado de pacientes','patient care') \
        .replace('evaluación de pacientes','patient evaluation') \
        .replace('extracciones de sangre','blood draw') \
        .replace('glucosa en sangre','blood glucose') \
        .replace('presión sanguínea','blood pressure') \
        .replace('medicina intensiva','intensive medicine') \
        .replace('control de infecciones','infection control') \
        .replace('enfermedades infecciosas','infectious diseases') \
        .replace('pruebas de función pulmonar','pulmonary function tests') \
        .replace('flujo de pacientes','patient flow') \
        .replace('historia clínica','medical record') \
        .replace('soporte vital','life support') \
        .replace('computación en la nube','cloud computing') \
        .replace('servicio de atención al cliente','customer service') \
        .replace('design da experiência do usuário','user experience design') \
        .replace('operaciones de redes informáticas','computer network operations') \
        .replace('gestión de relaciones con clientes','customer relationship management') \
        .replace('planificación de recursos empresariales','enterprise resource planning') \
        .replace('creación de oportunidades de negocio','creating business opportunities') \
        .replace('procesamiento de grandes volúmenes de datos','big data processing') \
        .replace('investigación y desarrollo','research development') \
        .replace('infraestructura de red','network infrastructure') \
        .replace('industria farmacéutica','pharmaceutical industry') \
        .replace('asistencia médica','medical assistance') \
        .replace('asistencia domicilio','home care') \
        .replace('cuidados intensivos','intensive care') \
        .replace('ingeniería de redes','network engineering') \
        .replace('minería de datos','data mining') \
        .replace('inteligência empresarial','business intelligence') \
        .replace('visualización de datos','data visualization') \
        .replace('capacidad de análisis','analysis capacity') \
        .replace('capacidad de respuesta','responsiveness') \
        .replace('análisis de requisitos','requirements analysis') \
        .replace('análisis de orina','urine analysis') \
        .replace('análisis de datos','data analysis') \
        .replace('ciencia de datos','data science') \
        .replace('aprendizaje automático','machine learning') \
        .replace('modelos matemáticos','mathematical models') \
        .replace('experiencia del cliente','customer experience') \
        .replace('eficacia de ventas','sales effectiveness') \
        .replace('makeup applicatior','makeup application') \
        .replace('control de proyectos','project control') \
        .replace('inglês fluente','fluent english') \
        .replace('industria petrolera','oil industry') \
        .replace('visual análisis de datos','visual data analysis') \
        .replace('ingeniería del petróleo','petroleum engineering') \
        .replace('instalación de software','software installation') \
        .replace('inteligencia artificial','artificial intelligence') \
        .replace('servicio de atención al cliente','customer service') \
        .replace('aptitudes de organización','organizational skills') \
        .replace('servidores de seguridad','security servers') \
        .replace('atendimento ao cliente','customer service') \
        .replace('atendimento ao paciente','patient service') \
        .replace('asistencia sanitaria','health care') \
        .replace('captação de clientes','customer acquisition') \
        .replace('recogida de especímenes','specimen collection') \
        .replace('definição de metas','definition goals') \
        .replace('gestão de projetos','project management') \
        .replace('gestão de conflitos','conflict management') \
        .replace('gestão de vendas','sales management') \
        .replace('gestión de redes sociales','social media management') \
        .replace('gestión de medios line','online media management') \
        .replace('gestión de redes','network management') \
        .replace('gestión operativa','operational management') \
        .replace('gestion de projet','project management') \
        .replace('gestión de proyectos','project management') \
        .replace('habilidades sociales','social skills') \
        .replace('planeamiento de proyectos','project planning') \
        .replace('desarrollo de proyectos','project development') \
        .replace('desarrollo de software','software development') \
        .replace('mejora continua','continuous improvement') \
        .replace('marketing de mídias sociais','social media marketing') \
        .replace('resolução de problemas','problem solving') \
        .replace('autorización previa','prior authorization') \
        .replace('atención telefónica','telephone assistance') \
        .replace('diseño de redes','network design') \
        .replace('diseño de moda','fashion design') \
        .replace('arte y manualidades','arts crafts') \
        .replace('industria cosmética','cosmetic industry') \
        .replace('trabalho em equipe','teamwork') \
        .replace('trabajo en equipo','teamwork') \
        .replace('espíritu de equipo','team spirit') \
        .replace('relación con el cliente','customer relationship') \
        .replace('solución de problemas','troubleshooting') \
        .replace('relaciones laborales','labor relations') \
        .replace('orientación objetivos','goal orientation') \
        .replace('optimización de procesos','process optimization') \
        .replace('comunicación inalámbrica','wireless communication') \
        .replace('asistencia directa al paciente','direct patient care') \
        .replace('publicidad en las redes sociales','advertising social networks') \
        .replace('eficacia organizacional','organizational effectiveness') \
        .replace('desenvolvimento de negócios','business development') \
        .replace('desenvolvimento de novos negócios','new business development') \
        .replace('desenvolvimento de software','software development') \
        .replace('desenvolvimento android','android development') \
        .replace('desenvolvimento de jogos eletrônicos','video game development') \
        .replace('desenvolvimento de backend','backend development') \
        .replace('unix aplicativos web','unix web applications') \
        .replace('linguagem de programação','programming language') \
        .replace('protocolos de internet','internet protocol') \
        .replace('redes informáticas','computer networks') \
        .replace('soporte técnico','technical support') \
        .replace('computação gráfica','computer graphics') \
        .replace('engenharia de software','software engineering') \
        .replace('requisitos de software','software requirements') \
        .replace('gestión de compras','purchasing management') \
        .replace('sector automovilístico','automotive sector') \
        .replace('procedimientos de oficina','office procedures') \
        .replace('manutenção preventiva','preventive maintenance') \
        .replace('asesoramiento académico','academic counseling') \
        .replace('planejamento de projetos','project planning') \
        .replace('páginas de visualforce','visualforce pages') \
        .replace('perfuração ao largo','offshore drilling') \
        .replace('sistemas embebidos','embedded systems') \
        .replace('espíritu empresarial','entrepreneurship') \
        .replace('pedidos de compra','purchase orders') \
        .replace('redes sociales','social networks') \
        .replace('maquillador','makeup artist') \
        .replace('electrónica','electronic') \
        .replace('liderazgo','leadership') \
        .replace('manufactura','manufacture') \
        .replace('pruebas','validation') \
        .replace('docência','teaching') \
        .replace('español','spanish') \
        .replace('inglês','english') \
        .replace('cardiologia','cardiology') \
        .replace('requisiciones','requisitions') \
        .replace('comunicação','communication') \
        .replace('comunicación','communication') \
        .replace('liderança','leadership') \
        .replace('presupuestos','budgets') \
        .replace('microcontroladores','microcontrollers') \
        .replace('empreendedorismo','entrepreneurship') \
        .replace('confidencialidad','confidentiality') \
        .replace('contabilidad','accounting') \
        .replace('recomendaciones','recommendations') \
        .replace('programación','programming') \
        .replace('fisioterapia','physiotherapy') \
        .replace('publicidad','advertising') \
        .replace('creatividad','creativity') \
        .replace('conmutadores','switches') \
        .replace('enrutadores','routers') \
        .replace('enrutamiento','routing') \
        .replace('microondas','microwave') \
        .replace('operaciones','operations') \
        .replace('espirometría','spirometry') \
        .replace('seguimiento','tracing') \
        .replace('estratégia','strategy') \
        .replace('formación','formation') \
        .replace('delegación','delegation') \
        .replace('calibração','calibration') \
        .replace('oratoria','oratory') \
        .replace('análisis','analysis') \
        .replace('compras','shopping') \
        .replace('mecanografia','typing') \
        .replace('redacción','drafting') \
        .replace('entrevistas','interviews') \
        .replace('negociación','negotiation') \
        .replace('inmunizaciones','immunizations') \
        .replace('mentoria','mentoring') \
        .replace('diseño','design') \
        .replace('ventas','sales') \
        .replace('inyecciones','injections') \
        .replace('clínicas','clinics') \
        .replace('triaje','triage') \
        .replace('vacunas','vaccines') \
        .replace('flebotomía','phlebotomy') \
        .replace('terminología','terminology') \
        .replace('hospitales','hospitals') \
        .replace('telecomunicaciones','telecommunications') \
        .replace('programmiersprache','programming language') \
        .replace('datenstrukturen','data structures') \
        .replace('softwareentwicklung','software development') \
        .replace('vrentwicklung','vr development') \
        .replace('algorithmens','algorithms') \
        .replace('bigdata et informatique décisionnelle','big data business intelligence') \
        .replace('régressions econométriques et analyse des données sur','economic regressions data analysis') \
        .replace('gestion de la relation avec les fournisseurs','supplier relationship management') \
        .replace('gestion de bases de données sur sql','database management sql') \
        .replace('développement pour android','development android') \
        .replace('gestion de risques','risk management') \
        .replace('apprentissage automatique','machine learning') \
        .replace('intelligence artificielle','artificial intelligence') \
        .replace('segmentation clientèle','customer segmentation') \
        .replace('reporting et performance sur','reporting performance') \
        .replace('réglémentations bancaires','banking regulations') \
        .replace('produits dérivés et pricing','derivatives pricing') \
        .replace('développement de logiciel','software development') \
        .replace('amélioration des processus','process improvement') \
        .replace('procesos de compra','purchasing process') \
        .replace('conception assistée par ordinateur','computeraided design') \
        .replace('mathématiques financières','financial mathematics') \
        .replace('génie mécanique','mechanical engineering') \
        .replace('méthode des éléments finis','finite element method') \
        .replace('budgétisation et prévision','budgeting forecasting') \
        .replace('résolution de problèmes','problem solving') \
        .replace('étude de marché','market research') \
        .replace('stratégie marketing','marketing strategy') \
        .replace('parler en public','public speaking') \
        .replace('travail déquipe','teamwork') \
        .replace('bâle à bâle','basel basel') \
        .replace('français','french') \
        .replace('programmation','programming') \
        .replace('planification','planning') \
        .replace('portfoliomanagement','portfolio management') \
        .replace('oplossen van problemen','troubleshooting') \
        .replace('financiële analyse','financial analysis') \
        .replace('financiële markten','financial market') \
        .replace('financiën','finance') \
        .replace('investeringen','investment') \
        .replace('ondernemerschap','entrepreneurship') \
        .replace('statistieken','statistics') \
        .replace('problemlösning','problem solving') \
        .replace('профессиональные навыки работы с компьютером','professional computer skills') \
        .replace('брокерская деятельность в области страховых услуг','brokerage activities insurance services') \
        .replace('работа с кредиторской задолженностью','work payable accounts') \
        .replace('гибкая методология программирования','agile programming methodology') \
        .replace('разработка мобильных приложений','mobile application development') \
        .replace('разработка приложений для','application development') \
        .replace('разработка программного обеспечения','software development') \
        .replace('управление предприятием','enterprise management') \
        .replace('финансовая отчётность','financial statements') \
        .replace('поиск и устранение неисправностей','troubleshooting') \
        .replace('промышленное производство','industrial production') \
        .replace('техническая документация','technical documentation') \
        .replace('непрерывная интеграция','continuous integration') \
        .replace('отношения с клиентами','customer relations') \
        .replace('управление проектами','project management') \
        .replace('критическое мышление','critical thinking') \
        .replace('машинное обучение','machine learning') \
        .replace('наука о данных','data science') \
        .replace('научная деятельность','scientific activity') \
        .replace('искусственный интеллект','artificial intelligence') \
        .replace('обработка изображений','image processing') \
        .replace('рекомендательные системы','recommendation system') \
        .replace('аналитические навыки','analytical skills') \
        .replace('английский язык','english language') \
        .replace('письменное сообщение','written message') \
        .replace('сбор данных','data collection') \
        .replace('структуры данных','data structures') \
        .replace('финансовый анализ','financial analysis') \
        .replace('финансовое моделирование','financial modeling') \
        .replace('фотографическое искусство','photographic art') \
        .replace('межличностное общение','interpersonal communication') \
        .replace('выверка счетов','account reconciliation') \
        .replace('заказы на  поставку','supply orders') \
        .replace('заказы на поставку','supply orders') \
        .replace('общее страхование','general insurance') \
        .replace('решение задач','problem solving') \
        .replace('сводные таблицы','pivot tables') \
        .replace('страхование жилища','home insurance') \
        .replace('страхование транспортных средств','vehicle insurance') \
        .replace('страхование от несчастных случаев','accident insurance') \
        .replace('конструктивная критика','constructive criticism') \
        .replace('страховые полисы','insurance policies') \
        .replace('контроль качества','quality control') \
        .replace('обслуживание клиентов','customer service') \
        .replace('венчурный капитал','venture capital') \
        .replace('корпоративные финансы','corporate finance') \
        .replace('частное инвестирование','private investment') \
        .replace('шаблоны проектирования','design patterns') \
        .replace('страхование','insurance') \
        .replace('финансы','finance') \
        .replace('электронные таблицы','spreadsheets') \
        .replace('инженерное дело','engineering') \
        .replace('аудит','audit') \
        .replace('продажи','sales') \
        .replace('скрам','scrum') \
        .replace('строительство','construction') \
        .replace('разработка по','development') \
        .replace('бухгалтерский учёт','accounting') \
        .replace('командная работа','teamwork') \
        .replace('розничная торговля','retail') \
        .replace('стратегия','strategy') \
        .replace('физика','physics') \
        .replace('алгоритмы','algorithms') \
        .replace('eksploracja danych','data mining') \
        .replace('procedury przechowywane','stored procedures') \
        .replace('rozwiązywanie problemów','problem solving') \
        .replace('umiejętności analityczne','analytical skills') \
        .replace('komunikacja','communication') \
        .replace('szczegółowość','detail') \
        .replace('统计数据分析','statistical data analysis') \
        .replace('趨勢科技','trending technology') \
        .replace('在线调研','online research') \
        .replace('無線網路','wireless network') \
        .replace('疑難排解','troubleshooting') \
        .replace('網路電話','internet telephone') \
        .replace('網際網路協定套組','internet protocol suite') \
        .replace('網際網路通訊協定','internet protocol') \
        .replace('公开演讲','public speaking') \
        .replace('數位媒體','digital media') \
        .replace('客戶關係管理','customer relationship management') \
        .replace('幾何尺寸與公差','geometric dimensions tolerances') \
        .replace('应用程序开发','application development') \
        .replace('智能软件','intelligent software') \
        .replace('社群媒體行銷','social media marketing') \
        .replace('有限元素分析','finite element analysis') \
        .replace('亚马逊网络服务系统','amazon web services') \
        .replace('实验室技能','laboratory skills') \
        .replace('团队建设','team building') \
        .replace('外語','foreign language') \
        .replace('人工智能','artificial intelligence') \
        .replace('人工智慧','artificial intelligence') \
        .replace('機器學習','machine learning') \
        .replace('机器学习','machine learning') \
        .replace('雙語溝通','bilingual communication') \
        .replace('演讲技能','presentation skills') \
        .replace('应用程序开发','application development') \
        .replace('分析技能','analytical skills') \
        .replace('计划管理','program management') \
        .replace('管理人员','management personnel') \
        .replace('个人发展','personal development') \
        .replace('产品管理','product management') \
        .replace('產品管理','product management') \
        .replace('项目管理','project management') \
        .replace('项目规划','project planning') \
        .replace('科研管理','research management') \
        .replace('数据结构','data structure') \
        .replace('数据分析','data analysis') \
        .replace('資料科學','data science') \
        .replace('社群媒體','social media') \
        .replace('社交媒体','social media') \
        .replace('金融建模','financial modeling') \
        .replace('產品設計','product design') \
        .replace('影像處理','image processing') \
        .replace('電腦視覺','computer vision') \
        .replace('智能软件','intelligent software') \
        .replace('人像摄影','portrait photography') \
        .replace('风险管理','risk management') \
        .replace('使用者體驗設計','user experience design') \
        .replace('用戶體驗','user experience') \
        .replace('实验性研究','experimental studies') \
        .replace('機械工程','mechanical engineering') \
        .replace('计量经济学','econometrics') \
        .replace('數位行銷','digital marketing') \
        .replace('云端平台','cloud platform') \
        .replace('领导力','leadership') \
        .replace('团队合作','teamwork') \
        .replace('英语','english') \
        .replace('研究','research') \
        .replace('药剂学','pharmacy') \
        .replace('软件','software') \
        .replace('医药','medicine') \
        .replace('生物','biology') \
        .replace('編輯','edit') \
        .replace('行銷','marketing') \
        .replace('演算法','algorithm') \
        .replace('算法','algorithm') \
        .replace('数据库','database') \
        .replace('程式設計','programming') \
        .replace('廣告','advertisement') \
        .replace('爱立信','ericsson') \
        .replace('沃尔玛','walmart') \
        .replace('英文','english') \
        .replace('语言','language') \
        .replace('摄影','photography') \
        .replace('工程','engineering') \
        .replace('機器人','robot') \
        .replace('设计','design') \
        .replace('통합팀관리','integrated team management') \
        .replace('리스크관리','risk management') \
        .replace('크리에이티브','creativity') \
        .replace('데이터입력','data entry') \
        .replace('문제해결','problem solving') \
        .replace('데이터분석','data analysis') \
        .replace('미디어기획','media planning') \
        .replace('사업개발','business development') \
        .replace('사업전략','business strategy') \
        .replace('소셜미디어','social media') \
        .replace('전략기획','strategic planning') \
        .replace('재무분석','financial analysis') \
        .replace('금융모델','financial modeling') \
        .replace('제품개발','product development') \
        .replace('재무감사','financial audit') \
        .replace('고객서비스','customer service') \
        .replace('팀워크','teamwork') \
        .replace('비즈니스','business') \
        .replace('오피스','office') \
        .replace('제조','manufacturing') \
        .replace('리더십','leadership') \
        .replace('예측','prediction') \
        .replace('한국어','korean') \
        .replace('영어','english') \
        .replace('온라인','online') \
        .replace('회계','accounting') \
        .replace('인터넷','internet') \
        .replace('광고','advertisement') \
        .replace('디지털','digital') \
        .replace('마케팅','marketing') \
        .replace('프로젝트','project') \
        .replace('재무','finance') \
        .replace('관리','management') \
        .replace('연설','speech') \
        .replace('감사','thanks') \
        .replace('소통','communication') \
        .replace('분석','analysis') \
        .replace('연금','pension') \
        .replace('통계','statistics') \
        .replace('अक्षित','akshit') \
        .replace('अरोड़ा','arora')

### Convert all textual values into list vectors word by word

In [9]:
### For proper conversion into lists, first all text characters must be lowercase and then this order is applied:
###   Text manipulation for many specific expressions/characters
###   Removal of emojis
###   Translation into English
###   Splitting words by whitespace
salaries_text_processed = (salaries['title'] + ' ' + salaries['main_skills'] + ' ' + salaries['skills']).str.lower()
row = 0
print('### ROW {} ###'.format(row))
print("BEFORE >>> \"{}\"\n".format(salaries_text_processed[row]))
s_time = time.time()
for i in range(len(salaries_text_processed)):
    text = salaries_text_processed.iloc[i]
    manipulated_text = manipulate_text(text)
    translated_text = translate_to_english(manipulated_text)
    salaries_text_processed.iloc[i] = translated_text
print("AFTER >>> \"{}\"\n".format(salaries_text_processed[row]))
salaries_text_processed = salaries_text_processed.apply(lambda x: x.split(' '))
print("VECTOR >>> {}\n".format(salaries_text_processed[row]))
print(">>> TOTAL TEXT PROCESSING TIME: {:.4f} seconds\n".format(time.time()-s_time))
# print("SET >>> {}".format(set(salaries_text_processed[row])))

### ROW 0 ###
BEFORE >>> "software developer ii at audible, inc. ajax, c, c++, css, data engineering, html, html/css, java, linux, microsoft office, mysql, oracle, oracle sql developer, pl/sql, programming, python, ruby on rails, software development, sql, sql scripting, xml ajax, c, c++, cascading style sheets (css), css, heroku, html, java, linux, microsoft office, mysql, oracle database, oracle sql developer, pl/sql, programming, python, ruby on rails, software development, sql, xml"

AFTER >>> "software developer audible incorporation ajax c cplusplus css data engineering html html css java linux microsoft office mysql oracle oracle sql developer plsql programming python rubyonrails software development sql sql scripting xml ajax c cplusplus cascading style sheets css css heroku html java linux microsoft office mysql oracle database oracle sql developer plsql programming python rubyonrails software development sql xml"

VECTOR >>> ['software', 'developer', 'audible', 'incorporation

### Check the list vectors of 3 randomly selected rows

In [10]:
# for i in range(3):
#     rand_row = random.randint(0,len(salaries_text_processed)-1)
#     print("### ROW {} ###\n{}".format(rand_row,salaries_text_processed.iloc[rand_row]))

### Create the dictionary structure

In [11]:
salaries_text_processed_sr = pd.Series(salaries_text_processed)
dictionary = gensim.corpora.Dictionary(salaries_text_processed_sr)
first_words = []
count = 0
# The first 75 words are shown in the output below
print("#### {} WORDS FOUND IN THE DICTIONARY ####".format(len(dictionary)))
print("------------- THE FIRST 75 WORDS -------------")
for k, v in dictionary.iteritems():
    first_words.append(v)
    count += 1
    if count > 74:
        break
for i in range(25):
    print("{:3} - {:24} | {:3} - {:24} | {:3} - {:24}".format(
        i, first_words[i], i+25, first_words[i+25], i+50, first_words[i+50]))

#### 15378 WORDS FOUND IN THE DICTIONARY ####
------------- THE FIRST 75 WORDS -------------
  0 - ajax                     |  25 - sheets                   |  50 - techniques              
  1 - audible                  |  26 - software                 |  51 - training                
  2 - c                        |  27 - sql                      |  52 - trends                  
  3 - cascading                |  28 - style                    |  53 - website                 
  4 - cplusplus                |  29 - xml                      |  54 - agile                   
  5 - css                      |  30 - associate                |  55 - analysis                
  6 - data                     |  31 - building                 |  56 - api                     
  7 - database                 |  32 - communication            |  57 - attention               
  8 - developer                |  33 - converse                 |  58 - automated               
  9 - development              |  

In [12]:
print("########  75 RANDOM WORDS IN THE DICTIONARY  ########")
random_words_ind = [];  random_words = []
for _ in range(75):
    rand_word = random.randint(0,len(dictionary)-1)
    random_words_ind.append(rand_word)
    random_words.append(dictionary[rand_word])
for i in range(25):
    print("{:5} - {:24} | {:5} - {:24} | {:5} - {:24}".format(
        random_words_ind[i], random_words[i], random_words_ind[i+25], random_words[i+25], random_words_ind[i+50], random_words[i+50]))

########  75 RANDOM WORDS IN THE DICTIONARY  ########
  745 - highspeed                | 10412 - fargate                  |  4305 - prototype               
14183 - csuf                     | 10753 - electrooptics            |  9593 - utdallas                
14751 - gatewaybcm               |  5785 - tidyverse                | 13111 - für                     
 3060 - tiktok                   | 11178 - pizza                    |  5171 - filemaker               
13555 - sdnnfv                   | 14609 - ltm                      |  8439 - desorptionpsd           
11382 - nng                      | 13368 - pigments                 |  3950 - theatre                 
 1589 - confluence               |  2959 - transformers             |  9417 - please                  
11840 - rootguard                | 14626 - diffractometry           | 14311 - coredata                
 1585 - ard                      |  8045 - arp                      |  9861 - eventbased              
  755 - semiconduct

### Search the target word in all list vectors (for finding the abnormals of such rows)

In [13]:
target_word = 'ipfs'
for i in range(len(salaries_text_processed)):
    if target_word in salaries_text_processed.iloc[i]:
        print(i, '->', salaries_text_processed.iloc[i])

2614 -> ['software', 'engineering', 'agile', 'bigdata', 'business', 'analysis', 'cloud', 'computing', 'documentation', 'financial', 'management', 'it', 'strategy', 'javascript', 'recruiting', 'recruitment', 'sdlc', 'software', 'development', 'stakeholder', 'management', 'strategy', 'team', 'management', 'technical', 'writing', 'agile', 'methodologies', 'bigdata', 'business', 'analysis', 'business', 'strategy', 'cloud', 'computing', 'crossfunctional', 'team', 'management', 'executive', 'coaching', 'fintech', 'insurtech', 'ipfs', 'it', 'strategy', 'javascript', 'pandas', 'software', 'process', 'improvement', 'optimization', 'recruiting', 'security', 'risk', 'management', 'software', 'development', 'life', 'cycle', 'sdlc', 'software', 'engineering', 'stakeholder', 'management', 'technical', 'documentation']


## Bag of Words

In [14]:
# For each document, this will create an individual dictionary reporting how many words and how many times those words appear.
# Save this to 'bow_corpus'
bow_corpus = [dictionary.doc2bow(s) for s in salaries_text_processed_sr]

In [15]:
# Preview bag of words for the randomly selected row
rand_row = random.randint(0,len(bow_corpus)-1)
print("### ROW", rand_row, "###")
print("VECTOR >>>", salaries_text_processed_sr[rand_row])
bow_doc_selected = bow_corpus[rand_row]
print("BoW CONVERSION >>>", bow_doc_selected)
for i in range(len(bow_doc_selected)):
    print("Word {} (\"{}\") appears {} {}.".format(
        bow_doc_selected[i][0], dictionary[bow_doc_selected[i][0]], bow_doc_selected[i][1], 
        "times" if bow_doc_selected[i][1] > 1 else "time"))

### ROW 2571 ###
VECTOR >>> ['software', 'engineer', 'paypal', 'android', 'programming', 'cplusplus', 'java', 'javascript', 'jquery', 'mongodb', 'mysql', 'oracle', 'php', 'python', 'android', 'development', 'cplusplus', 'coding', 'standards', 'crucible', 'java', 'javascript', 'jquery', 'jquery', 'mobile', 'mongodb', 'mysql', 'oracle', 'php', 'python', 'sonar', 'vera']
BoW CONVERSION >>> [(4, 2), (9, 1), (14, 2), (17, 2), (19, 2), (21, 1), (22, 2), (26, 1), (66, 1), (124, 2), (341, 1), (384, 2), (392, 3), (394, 2), (528, 1), (806, 2), (1101, 1), (2012, 1), (5974, 1), (5975, 1), (5976, 1)]
Word 4 ("cplusplus") appears 2 times.
Word 9 ("development") appears 1 time.
Word 14 ("java") appears 2 times.
Word 17 ("mysql") appears 2 times.
Word 19 ("oracle") appears 2 times.
Word 21 ("programming") appears 1 time.
Word 22 ("python") appears 2 times.
Word 26 ("software") appears 1 time.
Word 66 ("engineer") appears 1 time.
Word 124 ("javascript") appears 2 times.
Word 341 ("standards") appears 1

## Run LDA

In [16]:
# Train the LDA model with BoW using the related method from gensim and save it to 'lda_model_bow'
num_topics=25
s_time = time.time()
# gensim.models.LdaModel(corpus=None, num_topics=100, id2word=None, distributed=False, chunksize=2000, passes=1,
#     update_every=1, alpha='symmetric', eta=None, decay=0.5, offset=1.0, eval_every=10, iterations=50,
#     gamma_threshold=0.001, minimum_probability=0.01, random_state=None, ns_conf=None, minimum_phi_value=0.01,
#     per_word_topics=False, callbacks=None, dtype=<class 'numpy.float32'>)
lda_model_bow = gensim.models.LdaModel(corpus=bow_corpus, num_topics=num_topics, id2word=dictionary, distributed=False, 
                  chunksize=2000, passes=1, update_every=1, alpha='asymmetric', eta='auto', decay=0.5, 
                  offset=1.0, eval_every=10, iterations=25, gamma_threshold=0.1, minimum_probability=0.01,
                  minimum_phi_value=0.01, per_word_topics=False, random_state=42)
print("NUMBER OF TOPICS: {} | LDA using BoW lasted {:.3f} seconds.".format(num_topics, time.time() - s_time))

NUMBER OF TOPICS: 25 | LDA using BoW lasted 4.044 seconds.


### Here are the available methods to be used for LDA model

In [17]:
print(dir(lda_model_bow))

['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getitem__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_adapt_by_suffix', '_apply', '_load_specials', '_save_specials', '_smart_save', 'add_lifecycle_event', 'alpha', 'bound', 'callbacks', 'chunksize', 'clear', 'decay', 'diff', 'dispatcher', 'distributed', 'do_estep', 'do_mstep', 'dtype', 'eta', 'eval_every', 'expElogbeta', 'gamma_threshold', 'get_document_topics', 'get_term_topics', 'get_topic_terms', 'get_topics', 'id2word', 'inference', 'init_dir_prior', 'iterations', 'lifecycle_events', 'load', 'log_perplexity', 'minimum_phi_value', 'minimum_probability', 'num_terms', 'num_topics', 'num_updates', 'numworkers', 'offset', 'optimize_alpha', 'optimize_eta', 'passes', 'per_word_topics',

### Explore the words occurring for each topic and their relative weights

In [18]:
# print(dir(lda_model_bow))
print("TOPIC --> WORDS")
for idx, topic in lda_model_bow.print_topics(-1, 10):
    print('{:5} --> {}'.format(idx, topic))

TOPIC --> WORDS
    0 --> 0.179*"marketing" + 0.128*"sales" + 0.049*"strategy" + 0.048*"management" + 0.042*"media" + 0.038*"social" + 0.020*"knowledge" + 0.018*"techniques" + 0.017*"research" + 0.016*"campaigns"
    1 --> 0.167*"design" + 0.099*"user" + 0.055*"experience" + 0.040*"interface" + 0.040*"adobe" + 0.027*"web" + 0.024*"photoshop" + 0.023*"graphic" + 0.018*"creative" + 0.017*"ux"
    2 --> 0.085*"writing" + 0.051*"social" + 0.051*"media" + 0.038*"content" + 0.037*"editing" + 0.034*"research" + 0.021*"legal" + 0.020*"public" + 0.017*"marketing" + 0.014*"management"
    3 --> 0.178*"microsoft" + 0.059*"excel" + 0.059*"office" + 0.054*"management" + 0.047*"powerpoint" + 0.046*"word" + 0.037*"research" + 0.030*"public" + 0.029*"leadership" + 0.028*"speaking"
    4 --> 0.075*"network" + 0.060*"security" + 0.034*"computer" + 0.031*"networking" + 0.028*"windows" + 0.028*"networks" + 0.026*"administration" + 0.022*"configuration" + 0.022*"cisco" + 0.022*"troubleshooting"
    5 --> 0

## Number of rows classified for each topic

In [19]:
classified_rows_bow = {};  s_time = time.time()
for i in range(num_topics):
    classified_rows_bow[str(i)] = []
# Bag of Words
for i in range(len(salaries_text_processed_sr)):
    scores = sorted(lda_model_bow[bow_corpus[i]], key=lambda tup: -1*tup[1])
    classified_rows_bow[str(scores[0][0])].append(i)
print("-------- BoW --------")
for i in range(num_topics):
    print("{:2} --> {:5}".format(i, len(classified_rows_bow[str(i)])))
    # print(i, '-->', len(classified_rows_bow[str(i)]), classified_rows[str(i)])
    # print(i, '-->', len(classified_rows_tfidf[str(i)]), classified_rows[str(i)])
print("TIME ELAPSED FOR TOPIC ASSIGNMENTS: {:.4f} SECONDS".format(time.time() - s_time))

-------- BoW --------
 0 -->  1022
 1 -->   729
 2 -->   457
 3 -->  2231
 4 -->   954
 5 -->   254
 6 -->   369
 7 -->   360
 8 -->  2339
 9 -->   462
10 -->   103
11 -->  2453
12 -->   147
13 -->   143
14 -->   482
15 -->   215
16 -->  1371
17 -->  1149
18 -->  1193
19 -->   994
20 -->  1149
21 -->    14
22 -->    96
23 -->   590
24 -->   281
TIME ELAPSED FOR TOPIC ASSIGNMENTS: 3.5879 SECONDS


## Test LDA

In [20]:
rand_row = random.randint(0,len(bow_corpus)-1)
print("CHOSEN ROW: {}".format(rand_row))
print("VECTOR >>> {}".format(salaries_text_processed_sr[rand_row]))
print("BAG OF WORDS >>> ", bow_corpus[rand_row])
print("----------- BoW -----------")
scores_bow = sorted(lda_model_bow[bow_corpus[rand_row]], key=lambda tup: -1*tup[1])
print("SCORE                | TOPIC")
for index, score in scores_bow:
    print("{:<20} | {} --> {}".format(score, index, lda_model_bow.print_topic(index, 10)))

CHOSEN ROW: 6634
VECTOR >>> ['data', 'analyst', 'analytical', 'skills', 'business', 'analysis', 'c', 'data', 'analysis', 'data', 'engineering', 'databases', 'human', 'resources', 'microsoft', 'excel', 'microsoft', 'office', 'postgresql', 'powerpoint', 'programming', 'python', 'sql', 'sql', 'scripting', 'tableau', 'amazon', 'quicksight', 'amazon', 'redshift', 'amazon', 'sthree', 'analytical', 'skills', 'business', 'analysis', 'c', 'programming', 'language', 'data', 'analysis', 'databases', 'human', 'resources', 'hr', 'microsoft', 'excel', 'microsoft', 'office', 'microsoft', 'powerpoint', 'pivot', 'tables', 'postgresql', 'python', 'programming', 'language', 'sql', 'statistical', 'data', 'analysis', 'tableau']
BAG OF WORDS >>>  [(2, 2), (6, 5), (10, 1), (16, 5), (18, 2), (21, 3), (22, 2), (24, 1), (27, 3), (48, 2), (55, 5), (120, 3), (123, 2), (130, 1), (132, 2), (139, 2), (164, 1), (169, 2), (174, 2), (557, 2), (608, 2), (611, 2), (672, 1), (681, 2), (983, 2), (1465, 1), (1466, 1), (1860

## LDA Method Exploration

### __ getitem __

In [21]:
# __getitem__(bow, eps=None)
# Get the topic distribution for the given document
# Returns a topic distribution for the given document. Each topic is represented as a pair of its ID and the probability
# assigned to it.
lda_model_bow.__getitem__(bow_corpus[0])

[(3, 0.041476276), (7, 0.14769211), (20, 0.78900164)]

### bound

In [22]:
# bound(corpus, gamma=None, subsample_ratio=1.0)
# Estimate the variational bound of documents from the corpus as E_q[log p(corpus)] - E_q[log q(corpus)]
# Returns the variational bound score calculated for each document.
# This might take several seconds to complete.
lda_model_bow.bound(bow_corpus)

-7291194.45377183

### get_document_topics

In [23]:
# get_document_topics(bow, minimum_probability=None, minimum_phi_value=None, per_word_topics=False)
# Get the topic distribution for the given document
# Returns topic distribution for the whole document. Each element in the list is a pair of topic's ID, and the probability
# that was assigned to it. (here, the first 30 documents (rows) are printed)
dist = lda_model_bow.get_document_topics(bow_corpus)
for i in range(0,30):
    print(i, '->', dist[i])

0 -> [(3, 0.04098175), (7, 0.14733675), (20, 0.7951455)]
1 -> [(0, 0.15235561), (8, 0.8273668)]
2 -> [(5, 0.12894121), (8, 0.165836), (17, 0.66490644), (18, 0.022564504)]
3 -> [(9, 0.7055071), (17, 0.03352867), (23, 0.2431907)]
4 -> [(3, 0.4140923), (11, 0.12310646), (14, 0.071615696), (16, 0.22080445), (19, 0.06791148), (20, 0.024684377), (21, 0.04699401)]
5 -> [(3, 0.0127622), (4, 0.14723454), (8, 0.16095012), (10, 0.013681326), (13, 0.013362662), (14, 0.016751517), (16, 0.26282862), (17, 0.32437596), (22, 0.033942334)]
6 -> [(3, 0.16067871), (5, 0.022579683), (11, 0.20703982), (16, 0.5434689), (17, 0.05565901)]
7 -> [(3, 0.08405187), (5, 0.018610701), (14, 0.8841952)]
8 -> [(3, 0.384339), (5, 0.16451208), (9, 0.063844055), (11, 0.011362655), (18, 0.18983345), (19, 0.15358357), (21, 0.015470256)]
9 -> [(3, 0.64411235), (4, 0.131481), (7, 0.07572556), (8, 0.030694762), (14, 0.049414344), (16, 0.02903265), (24, 0.021876303)]
10 -> [(0, 0.08682905), (6, 0.11724333), (8, 0.07689763), (11

### get_term_topics

In [24]:
# get_term_topics(word_id, minimum_probability=None)
# Get the most relevant topics to the given word.
# Returns the relevant topics represented as pairs of their ID and their assigned probability, sorted by relevance
# to the given word (the first 100 words are printed, those that are not in the output have their weights pretty low)
for i in range(100):
    if len(lda_model_bow.get_term_topics(i)) != 0:
        print(i, '->', lda_model_bow.get_term_topics(i))

2 -> [(12, 0.021574171), (18, 0.011293029), (20, 0.018523455), (23, 0.088125385)]
4 -> [(11, 0.011828218), (12, 0.03925844), (18, 0.015537812), (19, 0.012682232), (20, 0.016533235), (23, 0.07625929)]
5 -> [(12, 0.04730838), (19, 0.029702082), (20, 0.071402155)]
6 -> [(3, 0.01303701), (6, 0.011296702), (7, 0.05063596), (10, 0.115448385), (11, 0.10083098), (16, 0.12466541), (18, 0.025397433), (19, 0.01109802), (20, 0.016895548), (21, 0.012029482), (24, 0.0146498345)]
7 -> [(7, 0.048656415)]
8 -> [(7, 0.011220019)]
9 -> [(1, 0.011218368), (13, 0.013353947), (15, 0.052061856), (17, 0.026566943), (18, 0.048567887), (19, 0.071351655), (20, 0.018842017), (24, 0.014486018)]
10 -> [(6, 0.044609837), (7, 0.011853456), (9, 0.024118554), (10, 0.016187236), (11, 0.01520819), (15, 0.011000781), (16, 0.012448197), (18, 0.013168522), (20, 0.019182317), (23, 0.012843639), (24, 0.012545925)]
12 -> [(12, 0.072307), (19, 0.031219317), (20, 0.09943204), (24, 0.013264111)]
14 -> [(10, 0.010405258), (12, 0.0

### get_topic_terms

In [25]:
# get_topic_terms(topicid, topn=10)
# Get the representation for a single topic. Words the integer IDs, in contrast to show_topic() that represents words
# by the actual strings.
# Returns Word ID - probability pairs for the most relevant words generated by the topic
lda_model_bow.get_topic_terms(0, topn=10)

[(41, 0.17857571),
 (46, 0.12792891),
 (49, 0.049402628),
 (40, 0.04810784),
 (270, 0.04185172),
 (364, 0.03774471),
 (201, 0.02027758),
 (50, 0.01772709),
 (128, 0.016641598),
 (407, 0.015834657)]

### get_topics

In [26]:
# get_topics()
# Get the term-topic matrix learned during inference
# Returns the probability for each word in each topic, shape (num_topics, vocabulary_size)
model_size = lda_model_bow.get_topics().shape
print("{} -> TOPICS: {}  VOCABULARY SIZE: {}".format(model_size, model_size[0], model_size[1]))
lda_model_bow.get_topics()

(25, 15378) -> TOPICS: 25  VOCABULARY SIZE: 15378


array([[1.5341471e-06, 6.1856372e-07, 4.0617149e-05, ..., 5.7264970e-07,
        5.7264975e-07, 5.7264970e-07],
       [9.2039882e-06, 8.5925683e-07, 1.2015456e-04, ..., 8.2466084e-07,
        8.2466084e-07, 8.2466084e-07],
       [2.7201829e-06, 2.4895191e-05, 3.6842201e-05, ..., 1.2544697e-06,
        1.2544697e-06, 1.2544697e-06],
       ...,
       [7.7317027e-06, 4.3754617e-06, 3.6806997e-04, ..., 4.2045513e-06,
        4.2045513e-06, 4.2045513e-06],
       [5.4690522e-06, 1.2435531e-06, 8.8108696e-02, ..., 1.2127933e-06,
        1.2127933e-06, 1.2127933e-06],
       [5.0506284e-03, 1.9707516e-06, 2.0711271e-03, ..., 1.9301267e-06,
        1.9301267e-06, 1.9301267e-06]], dtype=float32)

### print_topic

In [27]:
# print_topic(topicno, topn=10)
# Get a single topic as a formatted string.
# Returns string representation of topic.
chosen_topic = 0
print("--- TOPIC {} ---".format(chosen_topic))
print(lda_model_bow.print_topic(chosen_topic, topn=10))

--- TOPIC 0 ---
0.179*"marketing" + 0.128*"sales" + 0.049*"strategy" + 0.048*"management" + 0.042*"media" + 0.038*"social" + 0.020*"knowledge" + 0.018*"techniques" + 0.017*"research" + 0.016*"campaigns"


### print_topics

In [28]:
# print_topics(num_topics=20, num_words=10)
# Get the most significant topics (alias for show_topics() method).
# Returns sequence with (topic_id, [(word, value), ...])
lda_model_bow.print_topics(-1, num_words=6)

[(0,
  '0.179*"marketing" + 0.128*"sales" + 0.049*"strategy" + 0.048*"management" + 0.042*"media" + 0.038*"social"'),
 (1,
  '0.167*"design" + 0.099*"user" + 0.055*"experience" + 0.040*"interface" + 0.040*"adobe" + 0.027*"web"'),
 (2,
  '0.085*"writing" + 0.051*"social" + 0.051*"media" + 0.038*"content" + 0.037*"editing" + 0.034*"research"'),
 (3,
  '0.178*"microsoft" + 0.059*"excel" + 0.059*"office" + 0.054*"management" + 0.047*"powerpoint" + 0.046*"word"'),
 (4,
  '0.075*"network" + 0.060*"security" + 0.034*"computer" + 0.031*"networking" + 0.028*"windows" + 0.028*"networks"'),
 (5,
  '0.056*"information" + 0.046*"aws" + 0.038*"security" + 0.030*"technology" + 0.029*"amazon" + 0.025*"devops"'),
 (6,
  '0.087*"management" + 0.045*"project" + 0.045*"engineering" + 0.034*"autocad" + 0.025*"supply" + 0.024*"microsoft"'),
 (7,
  '0.097*"oracle" + 0.065*"sql" + 0.051*"data" + 0.049*"database" + 0.037*"sap" + 0.034*"scripting"'),
 (8,
  '0.133*"management" + 0.040*"sales" + 0.032*"customer"

### show_topic

In [29]:
# show_topic(topicid, topn=10)
# Get the representation for a single topic. Words here are the actual strings, in contrast to get_topic_terms() that
# represents words by their vocabulary ID.
# Returns word-probability pairs for the most relevant words generated by the topic.
chosen_topic = 0
print("--- TOPIC {} ---".format(chosen_topic))
print(lda_model_bow.show_topic(chosen_topic))

--- TOPIC 0 ---
[('marketing', 0.17857571), ('sales', 0.12792891), ('strategy', 0.049402628), ('management', 0.04810784), ('media', 0.04185172), ('social', 0.03774471), ('knowledge', 0.02027758), ('techniques', 0.01772709), ('research', 0.016641598), ('campaigns', 0.015834657)]


### show_topics

In [30]:
# show_topics(num_topics=10, num_words=10, log=False, formatted=True)
# Get a represenetation for selected topics.
# Returns a list of topics, each represented either as a string (when formatted == True) or word-probability pairs.
lda_model_bow.show_topics(-1, 6, formatted=True)

[(0,
  '0.179*"marketing" + 0.128*"sales" + 0.049*"strategy" + 0.048*"management" + 0.042*"media" + 0.038*"social"'),
 (1,
  '0.167*"design" + 0.099*"user" + 0.055*"experience" + 0.040*"interface" + 0.040*"adobe" + 0.027*"web"'),
 (2,
  '0.085*"writing" + 0.051*"social" + 0.051*"media" + 0.038*"content" + 0.037*"editing" + 0.034*"research"'),
 (3,
  '0.178*"microsoft" + 0.059*"excel" + 0.059*"office" + 0.054*"management" + 0.047*"powerpoint" + 0.046*"word"'),
 (4,
  '0.075*"network" + 0.060*"security" + 0.034*"computer" + 0.031*"networking" + 0.028*"windows" + 0.028*"networks"'),
 (5,
  '0.056*"information" + 0.046*"aws" + 0.038*"security" + 0.030*"technology" + 0.029*"amazon" + 0.025*"devops"'),
 (6,
  '0.087*"management" + 0.045*"project" + 0.045*"engineering" + 0.034*"autocad" + 0.025*"supply" + 0.024*"microsoft"'),
 (7,
  '0.097*"oracle" + 0.065*"sql" + 0.051*"data" + 0.049*"database" + 0.037*"sap" + 0.034*"scripting"'),
 (8,
  '0.133*"management" + 0.040*"sales" + 0.032*"customer"

### top_topics

In [31]:
# top_topics(corpus, texts=None, dictionary=None, window_size=None, coherence='u_mass', topn=20, processes=-1)
# Get the topics with the highest coherence score the coherence for each topic.
# Returns each element in the list as a pair of a topic representation and its coherence score. Topic representations are
# distributions of words, represented as a list of pairs of word IDs and their probabilities.
lda_model_bow.top_topics(bow_corpus, topn=10)

[([(0.099423274, 'html'),
   (0.071398266, 'css'),
   (0.06343832, 'java'),
   (0.063308716, 'sql'),
   (0.04934015, 'javascript'),
   (0.033803303, 'mysql'),
   (0.026786208, 'software'),
   (0.021331616, 'scripting'),
   (0.019412722, 'web'),
   (0.01921641, 'programming')],
  -0.7406374886407382),
 ([(0.10082312, 'data'),
   (0.07211259, 'learning'),
   (0.05480812, 'machine'),
   (0.04628004, 'python'),
   (0.033561785, 'sql'),
   (0.032071803, 'programming'),
   (0.027636606, 'analysis'),
   (0.02706289, 'r'),
   (0.025710752, 'language'),
   (0.024738908, 'science')],
  -0.9278860911009384),
 ([(0.17786895, 'microsoft'),
   (0.059020836, 'excel'),
   (0.05863274, 'office'),
   (0.05361004, 'management'),
   (0.047244765, 'powerpoint'),
   (0.04581818, 'word'),
   (0.03706299, 'research'),
   (0.029944658, 'public'),
   (0.02904288, 'leadership'),
   (0.028417608, 'speaking')],
  -0.9455399399214277),
 ([(0.07134656, 'development'),
   (0.043988038, 'javascript'),
   (0.039758176,

In [32]:
coherence_scores = []
top_topics = lda_model_bow.top_topics(bow_corpus, topn=10)
for i in range(len(top_topics)):
    coherence_scores.append(top_topics[i][1])
coherence_scores

[-0.7406374886407382,
 -0.9278860911009384,
 -0.9455399399214277,
 -0.9732423183323092,
 -0.9825687593145869,
 -0.989105210448775,
 -1.096832180155339,
 -1.1120127728739018,
 -1.1218903536878753,
 -1.2037612648136098,
 -1.262508078523438,
 -1.272832191008865,
 -1.3151714781356902,
 -1.4332698307190561,
 -1.4394553399724335,
 -1.4626667124480952,
 -1.4665507652228118,
 -1.6528731966606207,
 -1.653608762672896,
 -1.813003229300713,
 -1.9896994559523868,
 -2.465000417145571,
 -2.6920068034648246,
 -3.141876654004803,
 -9.29161572121641]